In [0]:
-- ╔════════════════════════════════════════════════════════════╗
-- ║  05_User_Silver_Cleaned                                   ║
-- ║  Silver_Cleaned User Table — Empty DDL Only               ║
-- ╚════════════════════════════════════════════════════════════╝
--
-- Source : yelp_poc_uc.bronze_final.yelp_user (~1,987,897 rows)
-- Targets (all in yelp_poc_uc.silver_cleaned):
--   yelp_user  (parent; 1 row per user)
--
-- ════════════════════════════════════════════════════════════
-- RULES & DESIGN PRINCIPLES
-- ════════════════════════════════════════════════════════════
-- (Complete definitions in 01_Business_Silver_Cleaned)
--
-- R1  DDL-Only / No Population
--     Table is created empty. Data population (type corrections,
--     CSV-to-array transformations) is handled by a separate
--     loading script.
--
-- R2  No Unnecessary Constraint Removal
--     PK/NOT NULL declared inline in CREATE TABLE.
--
-- R3  Normalization of Nested Data
--     Two bronze STRING columns contain embedded lists:
--       • elite  (CSV of years)    → ARRAY<INT>  (elite_years)
--       • friends (CSV of user IDs) → ARRAY<STRING> (friends)
--     Expanded into typed arrays rather than child tables,
--     because each list is a simple homogeneous collection.
--
-- R4  Referential Integrity via PK
--     PK on user_id declared inline.
--
-- R5  Semantic Data Types
--     Bronze Type       →  Silver Semantic Type  → Columns Affected
--     ────────────────────────────────────────────────────────────
--     STRING datetime   →  TIMESTAMP             → yelping_since
--     STRING CSV years  →  ARRAY<INT>            → elite_years
--     STRING CSV ids    →  ARRAY<STRING>          → friends
--     DOUBLE            →  DOUBLE (as-is avg)    → average_stars
--
-- R6  No Ingestion Timestamps
--     Lineage tracked via TBLPROPERTIES metadata.
--
-- R7  Self-Documenting DDL
--     Every column carries an inline comment.
--
-- Deferred FK Enablement:
--     After this table’s PK exists, the deferred FKs from
--     03_Review and 04_Tip are enabled via ALTER TABLE:
--       • yelp_review.user_id → yelp_user.user_id
--       • yelp_tip.user_id    → yelp_user.user_id
--
-- Execution order: Run LAST — enables deferred FKs from 03 & 04.
-- ════════════════════════════════════════════════════════════


-- ── 1. User Table (Silver Cleaned) ─────────────────────────
-- Core user profile data. One row per user. PK on user_id.
-- CSV columns (elite, friends) are expanded to typed arrays.

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_user (
  user_id            STRING    NOT NULL  COMMENT 'Unique 22-char Yelp user identifier. Primary key for the user profile table.',
  name               STRING              COMMENT 'User display name on Yelp.',
  review_count       BIGINT              COMMENT 'Total number of reviews written by this user.',
  yelping_since      TIMESTAMP           COMMENT 'When the user registered on Yelp. Source: bronze_final.yelp_user.yelping_since STRING cast to TIMESTAMP.',
  useful             BIGINT              COMMENT 'Total "useful" compliment votes received across all reviews.',
  funny              BIGINT              COMMENT 'Total "funny" compliment votes received across all reviews.',
  cool               BIGINT              COMMENT 'Total "cool" compliment votes received across all reviews.',
  elite_years        ARRAY<INT>          COMMENT 'Years the user held Yelp Elite status. Source: bronze_final.yelp_user.elite CSV STRING, split and cast to ARRAY<INT>. Empty array if never elite.',
  friends            ARRAY<STRING>       COMMENT 'Array of friend user_id hashes. Source: bronze_final.yelp_user.friends CSV STRING, split to ARRAY<STRING>. Empty array if no friends.',
  fans               BIGINT              COMMENT 'Number of fans/followers this user has.',
  average_stars      DOUBLE              COMMENT 'Average star rating across all user reviews. Computed average retained as DOUBLE.',
  compliment_hot     BIGINT              COMMENT 'Compliments received in "hot" category.',
  compliment_more    BIGINT              COMMENT 'Compliments received in "more" category.',
  compliment_profile BIGINT              COMMENT 'Compliments received in "profile" category.',
  compliment_cute    BIGINT              COMMENT 'Compliments received in "cute" category.',
  compliment_list    BIGINT              COMMENT 'Compliments received in "list" category.',
  compliment_note    BIGINT              COMMENT 'Compliments received in "note" category.',
  compliment_plain   BIGINT              COMMENT 'Compliments received in "plain" category.',
  compliment_cool    BIGINT              COMMENT 'Compliments received in "cool" compliment category.',
  compliment_funny   BIGINT              COMMENT 'Compliments received in "funny" compliment category.',
  compliment_writer  BIGINT              COMMENT 'Compliments received in "writer" category.',
  compliment_photos  BIGINT              COMMENT 'Compliments received in "photos" category.',
  CONSTRAINT pk_sc_user PRIMARY KEY (user_id)
)
USING DELTA
COMMENT 'Silver_Cleaned user profile table (DDL-only, no data). Type-corrected: yelping_since STRING→TIMESTAMP, elite CSV→ARRAY<INT>, friends CSV→ARRAY<STRING>. PK on user_id. ~1,987,897 rows when populated.'
TBLPROPERTIES (
  'quality'                = 'silver_cleaned',
  'pipelines.sourceTable'  = 'yelp_poc_uc.bronze_final.yelp_user',
  'design.populationMode'  = 'ddl_only'
);


-- ── 2. Enable Deferred Foreign Keys ─────────────────────────
--
-- Didnt run them till now(19-10-2026 11 PM)
--
-- Now that yelp_user has a PK, enable the FKs from Review and Tip
-- that were deferred in queries 03 and 04.

-- Review → User
-- ALTER TABLE yelp_poc_uc.silver_cleaned.yelp_review
--   DROP CONSTRAINT IF EXISTS fk_sc_review_user;

-- ALTER TABLE yelp_poc_uc.silver_cleaned.yelp_review
--   ADD CONSTRAINT fk_sc_review_user
--   FOREIGN KEY (user_id)
--   REFERENCES yelp_poc_uc.silver_cleaned.yelp_user (user_id);

-- -- Tip → User
-- ALTER TABLE yelp_poc_uc.silver_cleaned.yelp_tip
--   DROP CONSTRAINT IF EXISTS fk_sc_tip_user;

-- ALTER TABLE yelp_poc_uc.silver_cleaned.yelp_tip
--   ADD CONSTRAINT fk_sc_tip_user
--   FOREIGN KEY (user_id)
--   REFERENCES yelp_poc_uc.silver_cleaned.yelp_user (user_id);


-- ── 3. Verification ─────────────────────────────────────────

SELECT 'yelp_user' AS table_name, 'silver_cleaned' AS layer, COUNT(*) AS total_rows
FROM yelp_poc_uc.silver_cleaned.yelp_user;